## Data Ingestion

In [3]:
## document structure 
from langchain_core.documents import Document

In [4]:
from importlib.metadata import metadata
doc=Document(
    page_content="this is the main text content I am using to create RAG",
    metadata={
        "source":"example.txt",
        "pages":1,
        "author":"Romil Parikh",
        "date_created":"2026-06-08"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Romil Parikh', 'date_created': '2026-06-08'}, page_content='this is the main text content I am using to create RAG')

In [5]:
## create a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [6]:
import os

sample_texts = {
    "../data/text_files/python_intro.txt": """Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.

Created by Guido van Rossum and first released in 1991, Python has become one of the most popular programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.
""",

    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of Artificial Intelligence that enables systems to learn and improve from experience without being explicitly programmed.

It focuses on developing computer programs that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems.
"""
}

for filepath, content in sample_texts.items():
    # Create directory if it doesn't exist
    os.makedirs(os.path.dirname(filepath), exist_ok=True)

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created successfully!")

Sample text files created successfully!


In [7]:
from IPython.utils import encoding
## Text Loader
from langchain_community.document_loaders import TextLoader

from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)

C:\Users\HP\AppData\Local\Temp\ipykernel_13380\770491125.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\HP\Desktop\langchainupdated\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\n\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular programming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.\n')]


In [8]:
## Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## Load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## pattern to match files
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'},
    show_progress=False

)
documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of Artificial Intelligence that enables systems to learn and improve from experience without being explicitly programmed.\n\nIt focuses on developing computer programs that can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems.\n'),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\n\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular 

In [9]:
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader

## Load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)
pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'source': '..\\data\\pdf\\attention.pdf', 'file_path': '..\\data\\pdf\\attention.pdf', 'total_pages': 15, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '', 'modDate': 'D:20240410211143Z', 'creationDate': 'D:20240410211143Z', 'page': 0}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle

In [10]:
type(pdf_documents[0])

langchain_core.documents.base.Document

## RAG Pipelines - Data Ingestion to Vector DB pipelines

In [11]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [12]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory and its subdirectories"""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Check if directory exists
    if not pdf_dir.exists():
        print(f"Directory not found: {pdf_dir.resolve()}")
        return []

    # Find all PDF files recursively (case-insensitive)
    pdf_files = [
        file
        for file in pdf_dir.rglob("*")
        if file.is_file() and file.suffix.lower() == ".pdf"
    ]

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_path"] = str(pdf_file)
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)

            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")

    return all_documents


# Use your actual PDF folder
all_pdf_documents = process_all_pdfs(
    r"C:\Users\HP\Desktop\data\pdf"
)

print("\nFirst 5 documents:")
for doc in all_pdf_documents[:5]:
    print("-" * 50)
    print(doc.metadata)

Found 2 PDF files to process

Processing: attention.pdf


Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 44 0 (offset 0)
Ignoring wrong pointing object 60 0 (offset 0)
Ignoring wrong pointing object 112 0 (offset 0)
Ignoring wrong pointing object 120 0 (offset 0)


Loaded 15 pages

Processing: JP_final_21May26.pdf
Loaded 16 pages

Total documents loaded: 31

First 5 documents:
--------------------------------------------------
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Users\\HP\\Desktop\\data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_path': 'C:\\Users\\HP\\Desktop\\data\\pdf\\attention.pdf', 'file_type': 'pdf'}
--------------------------------------------------
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Ve

In [13]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Users\\HP\\Desktop\\data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_path': 'C:\\Users\\HP\\Desktop\\data\\pdf\\attention.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com

In [14]:
from sympy.solvers.diophantine.diophantine import length
## Text splitting get into chunks 

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks fr better RAG performace"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\n Example chunk:")
        print(f"content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata : {split_docs[0].metadata}")

    return split_docs

In [15]:

chunks=split_documents(all_pdf_documents)
chunks

Split 31 documents into 123 chunks

 Example chunk:
content: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...
Metadata : {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Users\\HP\\Desktop\\data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_path': 'C:\\Users\\HP\\Desktop\\data\\pdf\\attention.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Users\\HP\\Desktop\\data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_path': 'C:\\Users\\HP\\Desktop\\data\\pdf\\attention.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com

## Embedding and Vectorstroedb

In [16]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name:HuggingFace model name for sentence embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()


    def _load_model(self):
        """Load the sentenceTransformer model"""
        try:
            print(f"Loading embedding model:{self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimenson: {self.model.get_sentence_embedding_dimension()}")

        except Exception as e:
            print(f"Error loading model {self.model_name}:{e}")
            raise

    def generate_embeddings(self,texts:List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
             texts: List of text strings to embed

        Returns:
             numpy array of embeddings with shap (len(texts),embedding_dim)

        """
        if not self.model:
            raise ValueError("Model not loaded")


        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## Initialize the embeddings manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model:all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6318.11it/s]


Model loaded successfully. Embedding dimenson: 384


C:\Users\HP\AppData\Local\Temp\ipykernel_13380\4057464971.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimenson: {self.model.get_sentence_embedding_dimension()}")


## VectorStore

In [18]:
import os
import uuid
import chromadb
import numpy as np
from typing import List, Any

# Convert chunks to text

chunk_texts = [doc.page_content for doc in chunks]

print(f"Number of chunks: {len(chunk_texts)}")

# Generate embeddings

embeddings = embedding_manager.generate_embeddings(
    chunk_texts
)

print(f"Embeddings shape: {embeddings.shape}")
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:
            # Create directory if it doesn't exist
            os.makedirs(self.persist_directory, exist_ok=True)

            # Create persistent ChromaDB client
            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(
                f"Vector store initialized. Collection: {self.collection_name}"
            )
            print(
                f"Existing documents in collection: {self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """
        Add documents and embeddings to ChromaDB
        """

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(
            f"Adding {len(documents)} documents to vector store..."
        )

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(
                doc.page_content
            )

            metadatas.append(metadata)

            # Document text
            documents_text.append(
                doc.page_content
            )

            # Embedding
            embeddings_list.append(
                embedding.tolist()
            )

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} documents to vector store"
            )
            print(
                f"Total documents in collection: {self.collection.count()}"
            )

        except Exception as e:
            print(
                f"Error adding documents to vector store: {e}"
            )
            raise

    def get_document_count(self):
        return self.collection.count()

    def __repr__(self):
        return (
            f"VectorStore(collection='{self.collection_name}', "
            f"documents={self.collection.count()})"
        )


vectorstore = VectorStore()

vectorstore.add_documents(
    chunks,
    embeddings
)

print(
    f"Final Collection Count: "
    f"{vectorstore.collection.count()}"
)

vectorstore

Number of chunks: 123
Generating embeddings for 123 texts...


Batches: 100%|██████████| 4/4 [00:04<00:00,  1.23s/it]


Generated embeddings with shape: (123, 384)
Embeddings shape: (123, 384)
Vector store initialized. Collection: pdf_documents
Existing documents in collection: 738
Adding 123 documents to vector store...
Successfully added 123 documents to vector store
Total documents in collection: 861
Final Collection Count: 861


VectorStore(collection='pdf_documents', documents=861)

In [19]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Users\\HP\\Desktop\\data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_path': 'C:\\Users\\HP\\Desktop\\data\\pdf\\attention.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com

In [20]:
# converts the text to embeddings 
texts =[doc.page_content for doc in chunks]

embeddings=embedding_manager.generate_embeddings(texts)

vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 123 texts...


Batches: 100%|██████████| 4/4 [00:04<00:00,  1.18s/it]


Generated embeddings with shape: (123, 384)
Adding 123 documents to vector store...
Successfully added 123 documents to vector store
Total documents in collection: 984


## Retreiver Pipeline and VectorStore

In [21]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [22]:
rag_retriever

In [23]:
rag_retriever.retrieve("what is SUPERNEXA architecture?")

Retrieving documents for query: 'what is SUPERNEXA architecture?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 49.99it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_5a90b874_71',
  'content': 'include a Na’vi-inspired humanoid, a Toruk-inspired fantasy creature, a tiger-based non-human model, and a \nstylized cyberpunk female character. These characters were selected due to their significant differences in facial \ntopology and geometry, enabling evaluation of the framework’s ability to preserve facial expressions a cross \nhighly dissimilar target structures. Additionally, the framework was initialized using a pretrained facial expression \ndataset of 1000 images obtained from a publicly available Hugging Face repository for learning generalized facial \nmotion and expression dynamics[40]. \n \n3.2 Stages of the proposed SUPERNEXA Architecture \n \nThe SUPERNEXA stages are combination of geometric representation, motion modelling and deformation \nmechanisms. Fig. 2 shows the flowchart demonstrating the mathematical foundation of conversion of input facial',
  'metadata': {'creator': 'Microsoft Word',
   'moddate': "D:20260521141551Z

## RAG Pipeline - VectorDB to LLM output generation

In [24]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

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


In [25]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [26]:
class GroqLLM:
    def __init__(self,model_name:str="llama-3.3-70b-versatile",api_key:str=None):
        """
        Initialize Groq LLM

        Args:
             model_name:Groq model name (qwen2-72b-instruct , llama3-70b-8192,etc.)
             api_key:Groq API key (or set GROQ_API_KEY environment variable)

        """
        self.model_name=model_name
        self.api_key=api_key or os.environ.get("GROQ_API_KEY")

        if not self.api_key:
            raise ValueError("Groq API key is required.Set GROQ_API_KEY environment variable or pass api_key parameter.")

        self.llm=ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )

        print(f"Initialized Groq LLM wth model :{self.model_name}")

    def generate_response(self,query:str,context:str,max_length:int=500) -> str:
        """
        Generate response using retreievd context

        Args:
             query : User question
             context: Retrieved document context
             max_length : Maximum response length

        Returns:
             Generated response string
        """

        # Create prompt template
        prompt_template=PromptTemplate(
            input_variables=["context","question"],
            template="""You are a Helpful AI assistant.Use the following context to answer the question accurately and concisely.
            
Context:
{context}

Question:{question}

Answer : Provide a clear and informative answer based on the context above. If the context doesnt contain enough information to answer the question , say no.
"""
        )

        # format the prompt 
        formatted_prompt=prompt_template.format(context=context,question=query)

        try:

            #Generate response
            messages=[HumanMessage(content=formatted_prompt)]
            response=self.llm.invoke(messages)
            return response.content

        except Exception as e:
            return f"Error generating response:{str(e)}"

    def generate_response_simple(self,query:str,context:str)->str:
        """
        Simple response generation without complex prompting 

        Args:
             query:User question
             context:Retrieved context

            
        Returns:
            Generated response
        """

        simple_prompt=f"""Based on this context: {context}
        
Question:{query}

Answer:"""

        try:
            messages=[HumanMessage(content=simple_prompt)]
            response=self.llm.invoke(messages)
            return response.content

        except Exception as e:
            return f"Error : {str(e)}"


In [27]:
## Initialize Groq LLM 
try:
    groq_llm=GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except Exception as e:
    print(f"Warning : {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm=None

Initialized Groq LLM wth model :llama-3.3-70b-versatile
Groq LLM initialized successfully!


In [28]:
## get the context from the retriever and pass it to the LLM 
rag_retriever.retrieve("what is attention mechanism?")

Retrieving documents for query: 'what is attention mechanism?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 36.15it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_573dffe5_12',
  'content': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3',
  'metadata': {'total_pages': 15,
   'doc_index': 12,
   'producer': 'pdfTeX-1.40.25',
   'keywords': '',
   'creator': 'LaTeX with hyperref',
   'title': '',
   'page_label': '3',
   'creationdate': '2024-04-10T21:11:43+00:00',
   'file_type': 'pdf',
   'trapped': '/False',
   'source': 'C:\\Users\\HP\\Desktop\\data\\pdf\\attention.pdf',
   'moddate': '2024-04-10T21:11:43+00:00',
   'subject': '',
   'author': '',
   'content_length': 216,
   'file_path': 'C:\\Users\\HP\\Desktop\\data\\pdf\\attention.pdf',
   'source_file': 'attention.pdf',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'page': 2},
  'similarity_score': 0.271417498588562,
  'distance': 0.72858

## Integration Vectordb Context pipeline with LLM output

In [29]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content


In [32]:
answer=rag_simple("What is supernexa architecture?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is supernexa architecture?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 41.97it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The SUPERNEXA architecture is a combination of geometric representation, motion modeling, and deformation mechanisms for converting input facial expressions.


## Enhanced RAG Pipeline features

In [35]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Supernexa", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Supernexa'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 44.87it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [36]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 63.65it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

3.2 Attention
An attention functi

on can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

Question: what is attention is all you need

Answer:

Final Answer: "Attention is All You Need" refers to a paper that introduced the Transformer model, which relies entirely on attention mechanisms to process input sequences, eliminating the need for recurrent neural networks (RNNs) and convolutional neural networks (CNNs).

Citations:
[1] attention.pdf (page 2)
[2] attention.pdf (page 2)
[3] attention.pdf (page 2)
Summary: The paper "Attention is All You Need" introduced the Transformer model, a new approach to processing input sequences. The Transformer model relies solely on attention m